In [ ]:
#Graphes cycles



import os
import shutil

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

if os.path.exists('generated_graphs'):
    shutil.rmtree('generated_graphs')  # Supprime tout le dossier et son contenu
os.makedirs('generated_graphs', exist_ok=True)  # Recrée un répertoire vide





def generate_random_bipartite_graph(n):
    partition_size = n // 2
    remainder = n % 2

    G = nx.bipartite.random_graph(partition_size, partition_size + remainder, 0.5)
   
    adj_matrix = nx.to_numpy_array(G)
    if adj_matrix.shape != (n, n):
        adj_matrix = np.pad(adj_matrix, ((0, n - adj_matrix.shape[0]), (0, n - adj_matrix.shape[1])), mode='constant')
   
    return adj_matrix

def generate_random_planar_graph(n):
    side_length = int(np.ceil(np.sqrt(n)))
   
    G = nx.grid_2d_graph(side_length, side_length)
   
    # Convertir les nœuds de la grille en un seul index
    mapping = {node: i for i, node in enumerate(G.nodes())}
    G = nx.relabel_nodes(G, mapping)
   
    # Si le nombre de nœuds est supérieur à n, supprimer les nœuds supplémentaires
    if len(G.nodes()) > n:
        nodes_to_remove = list(G.nodes())[n:]
        G.remove_nodes_from(nodes_to_remove)
   
    # Ajouter des arêtes aléatoires tout en vérifiant que le graphe reste planaire
    possible_edges = [(u, v) for u in range(n) for v in range(u + 1, n) if not G.has_edge(u, v)]
    np.random.shuffle(possible_edges)
   
    for u, v in possible_edges:
        G.add_edge(u, v)
        if not nx.check_planarity(G)[0]:
            G.remove_edge(u, v)
   
    adj_matrix = nx.to_numpy_array(G)
   
    return adj_matrix

def generate_random_cyclic_graph(n):
    G = nx.Graph()  # Créer un graphe non orienté
    G.add_nodes_from(range(n))

    # Ajouter des arêtes pour créer un cycle
    edges = [(i, (i + 1) % n) for i in range(n)]  # Cycle de base
    G.add_edges_from(edges)

    # Ajouter des arêtes supplémentaires aléatoires pour augmenter la complexité
    while len(G.edges) < n + np.random.randint(1, n):
        u, v = np.random.choice(n, 2, replace=False)
        G.add_edge(u, v)

    return nx.to_numpy_array(G)

def generate_random_cycle_graph(n):
    G = nx.Graph()
    G.add_nodes_from(range(n))
   
    # Mélanger les sommets de manière aléatoire
    nodes = list(G.nodes())
    np.random.shuffle(nodes)
   
    # Ajouter des arêtes pour former un cycle
    edges = [(nodes[i], nodes[(i + 1) % n]) for i in range(n)]
    G.add_edges_from(edges)
   
    adj_matrix = nx.to_numpy_array(G)
   
    return adj_matrix

def generate_random_binary_tree(n):
    G = nx.DiGraph()  # Graphe orienté
    G.add_node(0)  # Ajouter la racine
    for i in range(1, n):
        parent = np.random.randint(0, i)  # Choisir un parent aléatoire
        # S'assurer que le parent n'a pas déjà deux enfants
        while G.out_degree(parent) >= 2:
            parent = np.random.randint(0, i)
        G.add_node(i)  # Ajouter le nœud
        G.add_edge(parent, i)  # Créer une arête orientée du parent vers l'enfant
    return nx.to_numpy_array(G)

def generate_random_tree(n):
    G = nx.Graph()
    G.add_nodes_from(range(n))
   
    # Ajouter des arêtes pour la connexité
    for i in range(1, n):
        u = np.random.randint(0, i)
        G.add_edge(u, i)
   
    # Ajouter des arêtes supplémentaires aléatoires pour augmenter la complexité
    num_edges = np.random.randint(n - 1, n * (n - 1) // 2)  # Nombre d'arêtes entre n-1 et n*(n-1)/2 (graphe complet)
    while G.number_of_edges() < num_edges:
        u, v = np.random.choice(n, 2, replace=False)
        if not G.has_edge(u, v):
            G.add_edge(u, v)
   
    T = nx.minimum_spanning_tree(G, algorithm='boruvka')
   
    adj_matrix = nx.to_numpy_array(T)
    if adj_matrix.shape != (n, n):
        adj_matrix = np.pad(adj_matrix, ((0, n - adj_matrix.shape[0]), (0, n - adj_matrix.shape[1])), mode='constant')
   
    return adj_matrix


# Enregistrer les graphes dans le dataset
def save_graphs(graph_dataset, n_graphs, n_nodes):
    for i in range(n_graphs):
        adj_matrix = graph_dataset[i].reshape(n_nodes, n_nodes)
        G = nx.from_numpy_array(adj_matrix)
       
        # Créer un fichier d'image pour chaque graphe
        if i % 10_000 == 0:
            plt.figure(figsize=(8, 6))
            try:
                nx.draw_planar(G, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
            except nx.NetworkXException:
                # Si le graphe n'est pas planaire, utiliser un autre layout
                nx.draw(G, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
           
            plt.title(f"Graphe {i + 1}")
            plt.show()

        # Enregistrer l'image dans le dossier 'generated_graphs'
        #plt.savefig(f'generated_graphs/graph_{i + 1}.png')
        plt.close()  # Fermer la figure pour libérer de la mémoire




class DegreeLayer(nn.Module):
    def __init__(self, n_nodes):
        super(DegreeLayer, self).__init__()
        self.n_nodes = n_nodes
        self.sum_layer = nn.Linear(n_nodes, 1, bias=False)
        # Set weights to 1 and freeze them
        self.sum_layer.weight.data.fill_(1)
        self.sum_layer.weight.requires_grad = False
       
    def forward(self, adj_matrix):
        batch_size = adj_matrix.size(0)
        degrees = torch.zeros(batch_size, self.n_nodes).to(adj_matrix.device)
        for i in range(self.n_nodes):
            degrees[:, i] = self.sum_layer(adj_matrix[:, i, :]).squeeze()
        return degrees

class Generator(nn.Module):
    def __init__(self, n_nodes):
        super(Generator, self).__init__()
        self.n_nodes = n_nodes
        self.fc = nn.Sequential(
            nn.Linear(n_nodes, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, n_nodes * n_nodes)
        )

    def forward(self, z):
        output = self.fc(z)
        adj_matrix = output.view(-1, self.n_nodes, self.n_nodes)
        adj_matrix = (adj_matrix + adj_matrix.transpose(1, 2)) / 2
        adj_matrix = torch.sigmoid(adj_matrix)
        adj_matrix = adj_matrix * (1 - torch.eye(self.n_nodes)).to(adj_matrix.device)
        #adj_matrix = (adj_matrix > 0.5).float()
        return adj_matrix

class Discriminator(nn.Module):
    def __init__(self, n_nodes):
        super(Discriminator, self).__init__()
        self.n_nodes = n_nodes
        self.fc = nn.Sequential(
            nn.Linear(n_nodes * n_nodes, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, adj_matrix):
        x = adj_matrix.view(-1, self.n_nodes * self.n_nodes)
        return self.fc(x)

def degree_loss(adj_matrix):
    degrees = torch.sum(adj_matrix, dim=2)
    return torch.sum(torch.sum((degrees - 2.0) ** 2, dim=1), dim=0)

def train(n_nodes=10, n_graphs=256, num_epochs=10_000, batch_size=32):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
   
    graph_dataset = [generate_random_cycle_graph(n_nodes) for _ in range(n_graphs)]
   
    generator = Generator(n_nodes).to(device)
    discriminator = Discriminator(n_nodes).to(device)
   
    g_optimizer = optim.Adam(generator.parameters(), lr=0.0001)
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0001)
    g_loss_list = []
    d_loss_list = []
    for epoch in range(num_epochs):
        for i in range(0, n_graphs, batch_size):
            real_graphs = torch.tensor(graph_dataset[i:i+batch_size], dtype=torch.float32).to(device)
           
            z = torch.randn(batch_size, n_nodes).to(device)
            fake_adj = generator(z)
           
            d_optimizer.zero_grad()
            d_real = discriminator(real_graphs)
            d_fake = discriminator(fake_adj.detach())
            d_loss = -torch.mean(torch.log(d_real) + torch.log(1 - d_fake))
            d_loss.backward()
            d_optimizer.step()
           
            g_optimizer.zero_grad()
            g_fake = discriminator(fake_adj)
            aux_loss = degree_loss(fake_adj)
            g_loss = -torch.mean(torch.log(1-g_fake)) + aux_loss
            g_loss.backward()
            g_optimizer.step()
            g_loss_list.append(g_loss.item())
            d_loss_list.append(d_loss.item())
         
        if epoch % 100 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], d_loss: {d_loss.item()}, g_loss: {g_loss.item()}, aux_loss: {aux_loss.item()}, g_without_aux_loss: {g_loss.item() - aux_loss.item()}')
    plt.plot(g_loss_list)
    plt.plot(d_loss_list)
    plt.show()
    torch.save(generator.state_dict(), 'generator_cycle.pth')
    torch.save(discriminator.state_dict(), 'discriminator_cycle.pth')
train(n_nodes=10, n_graphs=256, num_epochs=4_000, batch_size=32)

# Generate and visualize some graphs
def visualize_generated_graphs(generator, n_samples=5, n_nodes=10):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, n_nodes)
        fake_adj = (generator(z)>0.5).float()
       
        for i in range(n_samples):
            adj_matrix = fake_adj[i].numpy()
            G = nx.from_numpy_array(adj_matrix)
            degrees = torch.sum(fake_adj[i], dim=1).numpy().round(2)
           
            plt.figure(figsize=(8, 6))
            nx.draw(G, with_labels=True, node_color='lightblue',
                   node_size=500, font_size=16, font_weight='bold')
            plt.title(f'Generated Graph {i+1}\nNode Degrees: {degrees}')
            plt.show()

# Create a new generator instance
generator = Generator(n_nodes=10)
# Load the saved state
generator.load_state_dict(torch.load('generator_cycle.pth'))
# Call the visualization function
visualize_generated_graphs(generator)

In [ ]:
#Graphes arbres binaires

In [ ]:
#Graphes arbres binaires

In [ ]:
#Graphes planaires

In [ ]:
#Graphes cactus

def generate_cactus_graph(n, p):
    #Erdős-Rényi
    G = nx.erdos_renyi_graph(n, p)
    G = G.to_undirected()
    
    G = remove_common_edges(G)
    
    G = connect_disconnected_nodes(G)

    G = connect_degree_one_nodes(G, 1)
    
    return nx.adjacency_matrix(G).toarray()
def find_common_edges(G):
    cycles = list(nx.cycle_basis(G))
    edge_count = {}
    
    # Compter les occurrences de chaque arête dans les cycles
    for cycle in cycles:
        for i in range(len(cycle)):
            u, v = cycle[i], cycle[(i + 1) % len(cycle)]
            edge = tuple(sorted((u, v)))
            if edge in edge_count:
                edge_count[edge] += 1
                
            else:
                edge_count[edge] = 1
    return edge_count
def remove_common_edges(G):
    
    while True:
        # Supprimer les arêtes communes à deux cycles ou plus
        edge_count = find_common_edges(G)
        
        maxi = 0
        for edge, count in edge_count.items():
            if count > maxi:
                maxi = count


                
        
        if maxi > 1:
            #print(f"Removing edge {edge}")
            
            G.remove_edge(*edge)
        else:
            break
    
    return G

def connect_disconnected_nodes(G):
    # Ajouter une arête à tous les sommets non reliés pour s'assurer que le graphe est connexe
    components = list(nx.connected_components(G))
    while not nx.is_connected(G):
        u = random.choice(list(components[0]))
        v = random.choice(list(components[1]))
        G.add_edge(u, v)
        components = list(nx.connected_components(G))
    #print(len(components))
    
    return G

def connect_degree_one_nodes(G, pr):
    degree_one_nodes = [node for node, degree in G.degree() if degree == 1]
    
    num_to_keep = int(len(degree_one_nodes) * pr)
    
    selected_nodes = degree_one_nodes[:num_to_keep]
    random.shuffle(selected_nodes)
    
    # Créer des arêtes entre les sommets de la liste
    for i in range(0, len(selected_nodes) - 1, 2):
        u = selected_nodes[i]
        v = selected_nodes[i + 1]
        all_paths = list(nx.all_simple_paths(G, source=u, target=v))
        if len(all_paths) == 1:
            G.add_edge(u, v)
    
    return G

In [ ]:
# Generate 100 cactus graphs and check if they are cactus
cactus_count = 0
for _ in range(100):
    adj_matrix = generate_cactus_graph(n_nodes, p)
    G = nx.from_numpy_array(adj_matrix)
    if is_cactus(G):
        cactus_count += 1

print(f'Number of cactus graphs: {cactus_count} out of 100')
# Generate a connected simple graph that is not a cactus
non_cactus_graph = nx.Graph()
edges = [(1, 2), (2, 3), (3, 4), (4, 1), (1, 3)]
non_cactus_graph.add_edges_from(edges)

# Convert to adjacency matrix
non_cactus_adj_matrix = nx.to_numpy_array(non_cactus_graph)

# Check if the generated graph is a cactus
print(f'Is the generated graph a cactus? {is_cactus(non_cactus_graph)}')

In [ ]:
#code pour générer des cactus avec GAN


# Enregistrer les graphes dans le dataset
def save_graphs(graph_dataset, n_graphs, n_nodes):
    for i in range(n_graphs):
        adj_matrix = graph_dataset[i].reshape(n_nodes, n_nodes)
        G = nx.from_numpy_array(adj_matrix)
        
        # Créer un fichier d'image pour chaque graphe
        if i % 10_000 == 0:
            plt.figure(figsize=(8, 6))
            try:
                nx.draw_planar(G, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
            except nx.NetworkXException:
                # Si le graphe n'est pas planaire, utiliser un autre layout
                nx.draw(G, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
            
            plt.title(f"Graphe {i + 1}")
            plt.show()

        # Enregistrer l'image dans le dossier 'generated_graphs'
        #plt.savefig(f'generated_graphs/graph_{i + 1}.png')
        plt.close()  # Fermer la figure pour libérer de la mémoire




def simple_from_numpy_array(A):
    """Convert a 2D NumPy array to a NetworkX graph.

    Parameters
    ----------
    A : 2D numpy.ndarray
        An adjacency matrix representation of a graph.

    Returns
    -------
    G : NetworkX Graph
        A NetworkX graph generated from the adjacency matrix.
    """
    G = nx.Graph()
    n, m = A.shape
    if n != m:
        raise nx.NetworkXError("Adjacency matrix is not square.")
    
    G.add_nodes_from(range(n))
    edges = zip(*np.nonzero(A))
    G.add_edges_from((u, v, {'weight': A[u, v]}) for u, v in edges)
    
    return G
class Generator(nn.Module):
    def __init__(self, n_nodes):
        super(Generator, self).__init__()
        self.n_nodes = n_nodes
        self.fc = nn.Sequential(
            nn.Linear(n_nodes*n_nodes, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.7),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, n_nodes * n_nodes)
        )

    def forward(self, z):
        output = self.fc(z)
        adj_matrix = output.view(-1, self.n_nodes, self.n_nodes)
        adj_matrix = (adj_matrix + adj_matrix.transpose(1, 2)) / 2
        adj_matrix = torch.sigmoid(adj_matrix)
        adj_matrix = adj_matrix * (1 - torch.eye(self.n_nodes)).to(adj_matrix.device)
        adj_matrix = (adj_matrix > 0.5).float() + adj_matrix - adj_matrix.detach()
        return adj_matrix

class Discriminator(nn.Module):
    def __init__(self, n_nodes):
        super(Discriminator, self).__init__()
        self.n_nodes = n_nodes
        self.fc = nn.Sequential(
            nn.Linear(n_nodes * n_nodes, 128),
            nn.LeakyReLU(0.2),
            #nn.Dropout(0.4),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, adj_matrix):
        #print(adj_matrix.size())
        x = adj_matrix.view(-1, self.n_nodes * self.n_nodes)
        return self.fc(x)


def is_cactus(G):
    # A graph is a cactus if every edge belongs to at most one simple cycle
    if nx.number_connected_components(G) > 1:
        return False
    cycles = nx.cycle_basis(G)
    edge_count = {}
    for cycle in cycles:
        for i in range(len(cycle)):
            edge = tuple(sorted((cycle[i], cycle[(i + 1) % len(cycle)])))
            if edge in edge_count:
                return False
            edge_count[edge] = 1
    return True    
    
def cactus_loss(adj_matrix):
    batch_size, n_nodes, _ = adj_matrix.size()
    matrice = (adj_matrix>0.5).float() + adj_matrix - adj_matrix.detach()

    loss_cactus = torch.tensor(0.0)
    for i in range(batch_size):
        adj_matrix = matrice[i]
        G = nx.from_numpy_array(adj_matrix.cpu().detach().numpy())
        if is_cactus(G):
            loss_cactus += 1

    return batch_size-loss_cactus

def train(n_nodes=10, n_graphs=256, num_epochs=10_000, batch_size=32, fct=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(device)
    
    graph_dataset = [fct(n_nodes,p) for _ in range(n_graphs)]
    # Afficher le premier graphe du dataset
    first_graph_adj_matrix = graph_dataset[0]
    first_graph = nx.from_numpy_array(first_graph_adj_matrix)

    plt.figure(figsize=(8, 6))
    nx.draw(first_graph, with_labels=True, node_color="lightblue", font_weight="bold", node_size=700, edge_color="gray")
    plt.title("Premier Graphe du Dataset")
    plt.show()


    generator = Generator(n_nodes).to(device)
    discriminator = Discriminator(n_nodes).to(device)
    
    g_optimizer = optim.Adam(generator.parameters(), lr=0.00002)
    d_optimizer = optim.Adam(discriminator.parameters(), lr=0.002)
    cactus_list = []
    g_loss_list = []
    d_loss_list = []
    for epoch in range(num_epochs):
        #start_timeg = time.time()
        for i in range(0, n_graphs, batch_size):
            #real_graphs = torch.tensor(graph_dataset[i:i+batch_size], dtype=torch.float32).to(device)
            #start_timeg = time.time()
            batch_graphs_np = np.array(graph_dataset[i:i+batch_size])
            real_graphs = torch.tensor(batch_graphs_np, dtype=torch.float32).to(device)
            #print(graph_dataset[i])
            
            
            z = torch.randn(batch_size, n_nodes*n_nodes).to(device)
            
            fake_adj = generator(z)
            

            d_optimizer.zero_grad()
            d_real = discriminator(real_graphs)
            d_fake = discriminator(fake_adj.detach())
            d_loss = -torch.mean(torch.log(d_real) + torch.log(1 - d_fake))
            d_loss.backward()
            d_optimizer.step()
            
            g_optimizer.zero_grad()
            g_fake = discriminator(fake_adj)
            #gen_time = time.time() - start_timeg  
            #start_time = time.time()
            #custom_loss, n_moins_1 = aux_loss(fake_adj)
            #loss_time = time.time() - start_time
            
            g_loss = -torch.mean(torch.log(g_fake)) # + custom_loss
            g_loss_list.append(g_loss.cpu().detach().numpy())
            d_loss_list.append(d_loss.cpu().detach().numpy())
            g_loss.backward()
            g_optimizer.step()
        z = torch.randn(100, n_nodes*n_nodes).to(device)
        fake_adj = generator(z)
        cactus_count = 0
        for i in range(100):
            adj_matrix = fake_adj[i].cpu().detach().numpy()
            G = nx.from_numpy_array(adj_matrix)
            if is_cactus(G):
                cactus_count += 1
        cactus_percentage = cactus_count
        cactus_list.append(cactus_percentage)
            
        if epoch % 1 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], d_loss: {d_loss.item()}, g_loss: {g_loss.item()}, cactus_percentage: {cactus_percentage}%')
            #print(f'Time for generator: {gen_time:.4f}s, Time for tree_loss: {loss_time:.4f}s')
            torch.save(generator.state_dict(), 'generator_cycle2.pth')
            torch.save(discriminator.state_dict(), 'discriminator_cycle.pth')
    plt.plot(cactus_list)
    plt.show()
    plt.plot(g_loss_list)
    plt.plot(d_loss_list)
    plt.show()

n_nodes = 15
p=0.5
train(n_nodes=n_nodes, n_graphs=2048, num_epochs=200, batch_size=32, fct=generate_cactus_graph)

def visualize_generated_graphs(generator, n_samples=5, n_nodes=10):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, n_nodes*n_nodes)
        fake_adj = (generator(z) > 0.5).float()
        
        for i in range(n_samples):
            adj_matrix = fake_adj[i].numpy()
            G = nx.from_numpy_array(adj_matrix)
            degrees = torch.sum(fake_adj[i], dim=1).numpy().round(2)
            
            plt.figure(figsize=(8, 6))
            is_planar, _ = nx.check_planarity(G)
            if is_planar:
                nx.draw_planar(G, with_labels=True, node_color='lightblue', 
                               node_size=500, font_size=16, font_weight='bold')
            else:
                nx.draw(G, with_labels=True, node_color='lightblue', 
                        node_size=500, font_size=16, font_weight='bold')
            plt.title(f'Generated Graph {i+1}\nNode Degrees: {degrees}')
            plt.show()

# Exemple d'utilisation
n_nodes = 15
generator = Generator(n_nodes=n_nodes)
generator.load_state_dict(torch.load('generator_cycle2.pth'))

visualize_generated_graphs(generator, n_samples=10, n_nodes=n_nodes)